# Roach Academia — voice maker (CHIJOBHA's voice)

**One-time:** click the 🔑 (Secrets) icon on the left → **Add new secret** → Name `ROACH_KEY`, Value = your key → switch **Notebook access** on. The key is never saved in this file.

**Each time:** `Runtime → Change runtime type → T4 GPU`, then **`Runtime → Run all`**.

It takes the next chapter waiting for a voice, makes the audio in your voice (English and Telugu), and saves it to your private storage. Keep the tab open; it stops by itself when every waiting chapter is done. You can run it in several tabs or Google accounts at once — each takes a different chapter.


In [ ]:
try:
    from google.colab import userdata
    KEY = userdata.get("ROACH_KEY") or ""
except Exception:
    KEY = ""
assert KEY.strip(), "Add your key once: 🔑 Secrets (left bar) → Add new secret → Name ROACH_KEY → Notebook access ON. Then Run all again."
import subprocess
print(subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout or "NO GPU: Runtime → Change runtime type → T4 GPU")


In [2]:
# One-time setup (about 3 minutes): the voice engine (AI4Bharat IndicF5, MIT licence) and its model
import os, subprocess, hashlib
def sh(c): print(">>", c[:100]); subprocess.run(c, shell=True, check=True)
sh("pip install -q torchdiffeq x_transformers vocos ema_pytorch pydub jieba pypinyin cached_path soundfile librosa safetensors accelerate datasets wandb pyyaml")
if not os.path.isdir("/content/IndicF5-src"):
    sh("git clone -q --depth 1 https://github.com/AI4Bharat/IndicF5 /content/IndicF5-src")
sh("pip install -q --no-deps -e /content/IndicF5-src")
M = "/content/indicf5"; os.makedirs(M + "/checkpoints", exist_ok=True)
for f in ("config.json", "checkpoints/vocab.txt", "model.safetensors"):
    if not os.path.exists(f"{M}/{f}"):
        sh(f"curl -sSL --retry 4 -o {M}/{f} https://huggingface.co/Aditya02/IndicF5/resolve/main/{f}")
assert hashlib.sha256(open(M + "/model.safetensors", "rb").read()).hexdigest().startswith("ba7f3671180fb778"), "model download is damaged - Run all again"
print("Voice engine ready")


>> pip install -q torchdiffeq x_transformers vocos ema_pytorch pydub jieba pypinyin cached_path soundfi
>> git clone -q --depth 1 https://github.com/AI4Bharat/IndicF5 /content/IndicF5-src
>> pip install -q --no-deps -e /content/IndicF5-src
>> curl -sSL --retry 4 -o /content/indicf5/config.json https://huggingface.co/Aditya02/IndicF5/resolve/
>> curl -sSL --retry 4 -o /content/indicf5/checkpoints/vocab.txt https://huggingface.co/Aditya02/IndicF
>> curl -sSL --retry 4 -o /content/indicf5/model.safetensors https://huggingface.co/Aditya02/IndicF5/re
Voice engine ready


In [3]:
# Make voices until no chapter is waiting
import json, os, time, tarfile, shutil, subprocess, urllib.request, urllib.parse, uuid, datetime
FN = "https://xijkeywydoesxvqnacnc.supabase.co/functions/v1/video-admin"
def call(action, **q):
    r = urllib.request.Request(FN + "?action=" + action + "&" + urllib.parse.urlencode(q), method="POST", headers={"x-upload-token": KEY.strip()})
    return json.load(urllib.request.urlopen(r, timeout=120))
def get(key, path):
    urllib.request.urlretrieve(call("presign", method="GET", key=key, expires=3600)["url"], path)
def put(path, key):
    url = call("presign", method="PUT", key=key, expires=3600)["url"]
    urllib.request.urlopen(urllib.request.Request(url, data=open(path, "rb").read(), method="PUT"), timeout=900)
def rm(key): call("delete", key=key)
ME = uuid.uuid4().hex[:8]; now = lambda: datetime.datetime.now(datetime.timezone.utc)
done = 0
while True:
    items = call("list", prefix="work/colab/jobs/")["items"]
    keys = {i["key"]: i["at"] for i in items}
    jobs = sorted({k.split("/")[3] for k in keys if k.endswith("/payload.tgz")})
    pick = None
    for j in jobs:
        c = keys.get(f"work/colab/jobs/{j}/claim.json")
        if c and (now() - datetime.datetime.fromisoformat(c.replace("Z", "+00:00"))).total_seconds() < 3 * 3600: continue   # another tab has it
        pick = j; break
    if not pick:
        print(f"\nAll waiting chapters are voiced ({done} in this session). You can close this tab."); break
    base = f"work/colab/jobs/{pick}"
    open("/content/claim.json", "w").write(json.dumps({"by": ME})); put("/content/claim.json", f"{base}/claim.json")
    print(f"\n=== {pick} ===", flush=True); t0 = time.time()
    W = "/content/job"; shutil.rmtree(W, ignore_errors=True); os.makedirs(W)
    get(f"{base}/payload.tgz", "/content/payload.tgz"); tarfile.open("/content/payload.tgz").extractall(W)
    get(f"{base}/job.json", "/content/job.json"); job = json.load(open("/content/job.json"))
    env = dict(os.environ, INDICF5_DIR="/content/indicf5")
    for k, v in json.load(open(f"{W}/env.json")).items():
        if v: env[k] = (W + "/" + v) if k == "TE_REF" else v
    out = "/content/voice-out"; shutil.rmtree(out, ignore_errors=True)
    subprocess.run(["python", "tools/voice_te.py", "sentences.json", out], cwd=W, env=env, check=True)
    want = sum(len(v) for v in json.load(open(f"{W}/sentences.json", encoding="utf-8")).values())
    got = len([f for f in os.listdir(out) if f.endswith(".wav")])
    assert got >= want, f"made {got} of {want} sentences - Run all again to retry"
    subprocess.run(f"tar -czf /content/done.tgz -C {out} .", shell=True, check=True)
    put("/content/done.tgz", job["req"]["colab_voice"])
    open("/content/ready.json", "w").write(json.dumps(job["req"])); put("/content/ready.json", f"work/colab/ready/{pick}.json")
    for k in ("payload.tgz", "job.json", "claim.json"): rm(f"{base}/{k}")
    done += 1; print(f"=== {pick}: {got} sentences in {int(time.time() - t0)} s — video will be made automatically ===", flush=True)



=== promo-en-ad01s ===


/tmp/ipykernel_5631/863038094.py:30: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  get(f"{base}/payload.tgz", "/content/payload.tgz"); tarfile.open("/content/payload.tgz").extractall(W)


=== promo-en-ad01s: 8 sentences in 203 s — video will be made automatically ===

=== promo-te-g04s ===


/tmp/ipykernel_5631/863038094.py:30: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  get(f"{base}/payload.tgz", "/content/payload.tgz"); tarfile.open("/content/payload.tgz").extractall(W)


=== promo-te-g04s: 7 sentences in 208 s — video will be made automatically ===

All waiting chapters are voiced (2 in this session). You can close this tab.
